# Custom Object Detection with YOLO26m on COCO 2017

This notebook refactors Project 24 into a real notebook-first object detection workflow.

## 1. Project Overview

Goal: train a configurable custom-object detector by selecting a subset of COCO classes and fine-tuning YOLO26m.

## 2. Problem Statement

Detect only user-selected custom classes in images, then evaluate with proper detection metrics and qualitative predictions.

## 3. Why This Method Is Correct

Task family is object detection, so YOLO26m is the correct April 2026 default. COCO 2017 provides real bounding-box annotations and broad object coverage for custom class subsets.

## 4. Dataset Source and License Notes

Dataset source: https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset

Use this dataset according to Kaggle dataset terms and COCO licensing notes from original dataset owners.

## 5. Environment Setup

In [1]:
import os
import platform
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Python: 3.13.12
PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 6. Dependency Installation

In [2]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure_package('ultralytics')
ensure_package('kagglehub')
ensure_package('cv2', 'opencv-python')
ensure_package('matplotlib')
ensure_package('pandas')
print('Dependencies are available.')

e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(
e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencies are available.


## 7. Imports and Configuration

In [3]:
import json
import shutil
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

PROJECT_DIR = Path.cwd()
WORK_DIR = PROJECT_DIR / 'working'
DATA_ROOT = WORK_DIR / 'data'
YOLO_ROOT = DATA_ROOT / 'custom_coco_yolo'
RUNS_DIR = PROJECT_DIR / 'runs'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'

for d in [WORK_DIR, DATA_ROOT, YOLO_ROOT, RUNS_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = 'awsaf49/coco-2017-dataset'
CUSTOM_CLASSES = ['person', 'car', 'dog']
MAX_TRAIN_IMAGES = 2400
MAX_VAL_IMAGES = 400
MAX_TEST_IMAGES = 250

print(f'Project dir: {PROJECT_DIR}')
print(f'Custom classes: {CUSTOM_CLASSES}')

Project dir: e:\Github\Machine-Learning-Projects\Computer Vision\Project 24 - Custom Object Detection
Custom classes: ['person', 'car', 'dog']


## 8. Dataset Download and Loading

The next cell downloads COCO 2017 using Kaggle credentials in-notebook.

In [ ]:
def check_kaggle_credentials():
    has_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
    has_file = (Path.home() / '.kaggle' / 'kaggle.json').exists()
    if not has_env and not has_file:
        raise RuntimeError('Kaggle credentials not found. Set KAGGLE_USERNAME/KAGGLE_KEY or create ~/.kaggle/kaggle.json')

def download_coco():
    check_kaggle_credentials()
    try:
        import kagglehub
        return Path(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception:
        target = DATA_ROOT / 'coco_2017_raw'
        target.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', str(target), '--unzip'])
        return target

raw_root = Path(download_coco())
print(f'Raw COCO path: {raw_root}')

## 9. Data Validation Checks

In [ ]:
def find_coco_root(root):
    candidates = [root]
    if root.exists():
        for child in root.iterdir():
            if child.is_dir():
                candidates.append(child)
    for candidate in candidates:
        t1 = candidate / 'train2017'
        v1 = candidate / 'val2017'
        a1 = candidate / 'annotations'
        t2 = candidate / 'images' / 'train2017'
        v2 = candidate / 'images' / 'val2017'
        if t1.exists() and v1.exists() and a1.exists():
            return candidate
        if t2.exists() and v2.exists() and a1.exists():
            return candidate
    raise RuntimeError(f'Could not find COCO structure under: {root}')

COCO_ROOT = find_coco_root(raw_root)
TRAIN_IMG_ROOT = COCO_ROOT / 'train2017'
VAL_IMG_ROOT = COCO_ROOT / 'val2017'
if not TRAIN_IMG_ROOT.exists():
    TRAIN_IMG_ROOT = COCO_ROOT / 'images' / 'train2017'
if not VAL_IMG_ROOT.exists():
    VAL_IMG_ROOT = COCO_ROOT / 'images' / 'val2017'
TRAIN_JSON = COCO_ROOT / 'annotations' / 'instances_train2017.json'
VAL_JSON = COCO_ROOT / 'annotations' / 'instances_val2017.json'

assert TRAIN_IMG_ROOT.exists(), f'Missing train images: {TRAIN_IMG_ROOT}'
assert VAL_IMG_ROOT.exists(), f'Missing val images: {VAL_IMG_ROOT}'
assert TRAIN_JSON.exists(), f'Missing train json: {TRAIN_JSON}'
assert VAL_JSON.exists(), f'Missing val json: {VAL_JSON}'

print(f'COCO root: {COCO_ROOT}')
print(f'Train images: {TRAIN_IMG_ROOT}')
print(f'Val images: {VAL_IMG_ROOT}')

## 10. Label and Target Analysis

In [ ]:
with open(TRAIN_JSON, 'r', encoding='utf-8') as f:
    train_meta = json.load(f)
with open(VAL_JSON, 'r', encoding='utf-8') as f:
    val_meta = json.load(f)

categories = {c['id']: c['name'] for c in train_meta['categories']}
name_to_coco_id = {v: k for k, v in categories.items()}

missing = [name for name in CUSTOM_CLASSES if name not in name_to_coco_id]
if missing:
    raise RuntimeError(f'Custom classes not present in COCO categories: {missing}')

selected_coco_ids = [name_to_coco_id[name] for name in CUSTOM_CLASSES]
selected_map = {cid: i for i, cid in enumerate(selected_coco_ids)}

print('Selected class mapping (COCO -> YOLO):')
for cid in selected_coco_ids:
    print(f'  {cid} ({categories[cid]}) -> {selected_map[cid]}')

## 11. Train/Validation/Test Split Strategy

We use official COCO train split for training and partition COCO val split into validation and held-out test subsets.

In [ ]:
def build_index(meta):
    images_by_id = {img['id']: img for img in meta['images']}
    anns_by_image = defaultdict(list)
    for ann in meta['annotations']:
        if ann.get('iscrowd', 0) == 1:
            continue
        if ann['category_id'] not in selected_coco_ids:
            continue
        x, y, w, h = ann['bbox']
        if w <= 1 or h <= 1:
            continue
        anns_by_image[ann['image_id']].append(ann)
    image_ids = sorted(anns_by_image.keys())
    return images_by_id, anns_by_image, image_ids

train_images_by_id, train_anns_by_image, train_ids_all = build_index(train_meta)
val_images_by_id, val_anns_by_image, val_ids_all = build_index(val_meta)

train_ids = train_ids_all[:min(MAX_TRAIN_IMAGES, len(train_ids_all))]
val_ids = val_ids_all[:min(MAX_VAL_IMAGES, len(val_ids_all))]
test_start = len(val_ids)
test_ids = val_ids_all[test_start:test_start + min(MAX_TEST_IMAGES, len(val_ids_all) - test_start)]

if len(train_ids) == 0 or len(val_ids) == 0 or len(test_ids) == 0:
    raise RuntimeError('Insufficient images found for requested custom class subset split sizes.')

print(f'Train images: {len(train_ids)}')
print(f'Val images: {len(val_ids)}')
print(f'Test images: {len(test_ids)}')

## 12. Preprocessing and COCO-to-YOLO Conversion

In [ ]:
def write_split(split_name, image_root, images_by_id, anns_by_image, selected_ids):
    out_img = YOLO_ROOT / split_name / 'images'
    out_lbl = YOLO_ROOT / split_name / 'labels'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    for image_id in selected_ids:
        img = images_by_id[image_id]
        src_img = image_root / img['file_name']
        dst_img = out_img / img['file_name']
        if not dst_img.exists():
            shutil.copy2(src_img, dst_img)

        width = img['width']
        height = img['height']
        lines = []
        for ann in anns_by_image[image_id]:
            x, y, w, h = ann['bbox']
            cx = (x + w / 2.0) / width
            cy = (y + h / 2.0) / height
            nw = w / width
            nh = h / height
            cls = selected_map[ann['category_id']]
            lines.append(f'{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

        (out_lbl / f"{Path(img['file_name']).stem}.txt").write_text('\n'.join(lines), encoding='utf-8')

write_split('train', TRAIN_IMG_ROOT, train_images_by_id, train_anns_by_image, train_ids)
write_split('val', VAL_IMG_ROOT, val_images_by_id, val_anns_by_image, val_ids)
write_split('test', VAL_IMG_ROOT, val_images_by_id, val_anns_by_image, test_ids)

data_yaml = YOLO_ROOT / 'data.yaml'
data_yaml.write_text(
    '\n'.join([
        f'path: {YOLO_ROOT.as_posix()}',
        'train: train/images',
        'val: val/images',
        'test: test/images',
        f'nc: {len(CUSTOM_CLASSES)}',
        f'names: {CUSTOM_CLASSES}',
    ]),
    encoding='utf-8'
)

print(data_yaml.read_text(encoding='utf-8'))

## 13. Dataset Verification After Conversion

In [ ]:
def count_files(path):
    return len([p for p in path.iterdir() if p.is_file()])

for split in ['train', 'val', 'test']:
    img_dir = YOLO_ROOT / split / 'images'
    lbl_dir = YOLO_ROOT / split / 'labels'
    assert img_dir.exists(), f'Missing images directory: {img_dir}'
    assert lbl_dir.exists(), f'Missing labels directory: {lbl_dir}'
    n_img = count_files(img_dir)
    n_lbl = count_files(lbl_dir)
    assert n_img == n_lbl, f'Mismatch in {split}: images={n_img}, labels={n_lbl}'
    print(f'{split}: images={n_img}, labels={n_lbl}')

## 14. Exploratory Sample Visualization

In [ ]:
sample_paths = sorted((YOLO_ROOT / 'train' / 'images').iterdir())[:9]
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, image_path in zip(axes.flatten(), sample_paths):
    label_path = YOLO_ROOT / 'train' / 'labels' / f'{image_path.stem}.txt'
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lines = label_path.read_text(encoding='utf-8').strip().splitlines()
    for line in lines:
        cls, cx, cy, nw, nh = line.split()
        cls = int(cls)
        cx, cy, nw, nh = map(float, [cx, cy, nw, nh])
        bw = int(nw * w)
        bh = int(nh * h)
        x1 = int(cx * w - bw / 2)
        y1 = int(cy * h - bh / 2)
        x2 = x1 + bw
        y2 = y1 + bh
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 50, 50), 2)
        cv2.putText(img, CUSTOM_CLASSES[cls], (x1, max(0, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 50, 50), 2)
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'sample_boxes.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Baseline and Main Training Workflow

We train YOLO26m first and fall back to YOLO26s on OOM.

In [ ]:
EPOCHS = 20
IMGSZ = 960
BATCH = 8 if DEVICE == 'cuda' else 4
PRIMARY = 'yolo26m.pt'
FALLBACK = 'yolo26s.pt'

selected_model = PRIMARY
model = YOLO(selected_model)
try:
    model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RUNS_DIR),
        name='custom_object_detection',
        exist_ok=True,
        device=0 if DEVICE == 'cuda' else 'cpu',
        seed=SEED,
        workers=2,
        pretrained=True,
        verbose=True,
    )
except RuntimeError as exc:
    if 'out of memory' not in str(exc).lower():
        raise
    selected_model = FALLBACK
    model = YOLO(selected_model)
    model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=max(2, BATCH // 2),
        project=str(RUNS_DIR),
        name='custom_object_detection',
        exist_ok=True,
        device=0 if DEVICE == 'cuda' else 'cpu',
        seed=SEED,
        workers=2,
        pretrained=True,
        verbose=True,
    )

run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.exists(), f'Missing best weights: {best_pt}'
print(f'Selected model: {selected_model}')
print(f'Run dir: {run_dir}')

## 16. Evaluation

Detection metrics reported: precision, recall, mAP50, mAP50-95.

In [ ]:
best_model = YOLO(str(best_pt))
val_result = best_model.val(data=str(data_yaml), split='test', imgsz=IMGSZ, batch=max(1, BATCH // 2), device=0 if DEVICE == 'cuda' else 'cpu', verbose=False)

metrics = {
    'dataset': KAGGLE_DATASET,
    'custom_classes': CUSTOM_CLASSES,
    'selected_model': selected_model,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'train_images': len(train_ids),
    'val_images': len(val_ids),
    'test_images': len(test_ids),
    'precision': float(val_result.box.mp),
    'recall': float(val_result.box.mr),
    'map50': float(val_result.box.map50),
    'map50_95': float(val_result.box.map),
}

metrics_path = ARTIFACTS_DIR / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))

## 17. Qualitative Inference Examples

In [ ]:
test_images = sorted((YOLO_ROOT / 'test' / 'images').iterdir())[:6]
pred_frames = []
for image_path in test_images:
    pred = best_model.predict(source=str(image_path), conf=0.25, verbose=False)[0]
    pred_frames.append(cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB))

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, frame in zip(axes.flatten(), pred_frames):
    ax.imshow(frame)
    ax.axis('off')
for ax in axes.flatten()[len(pred_frames):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'qualitative_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 18. Error Analysis and Limitations

Likely failure modes:
- small or distant objects
- class ambiguity in crowded scenes
- domain mismatch between deployment images and COCO

This notebook evaluates detector performance, not tracking performance.

## 19. How To Improve

- Increase subset size for better class coverage
- Tune augmentation and image size for small objects
- Add harder negative examples for selected classes
- Compare YOLO26m vs YOLO26l for accuracy/latency trade-off

## 20. Production Considerations

- Export ONNX and benchmark target hardware latency
- Use class-specific confidence thresholds if needed
- Keep a held-out domain validation set for deployment drift checks

## 21. Common Mistakes

- training with class names not present in COCO
- mixing train and validation images (data leakage)
- evaluating only qualitative images without metrics
- forgetting to verify image/label count parity

## 22. Mini Challenge

Change `CUSTOM_CLASSES` to any 2-4 classes of your choice, regenerate YOLO labels, and compare metric changes against the current run.

## 23. Artifact Saving and Final Summary

In [ ]:
onnx_path = Path(best_model.export(format='onnx'))
final_best_pt = ARTIFACTS_DIR / 'best.pt'
final_best_onnx = ARTIFACTS_DIR / 'best.onnx'
shutil.copy2(best_pt, final_best_pt)
shutil.copy2(onnx_path, final_best_onnx)

manifest = {
    'project': 'Custom Object Detection',
    'dataset': KAGGLE_DATASET,
    'custom_classes': CUSTOM_CLASSES,
    'selected_model': selected_model,
    'artifacts': {
        'best_pt': str(final_best_pt),
        'best_onnx': str(final_best_onnx),
        'metrics_json': str(metrics_path),
        'sample_boxes_png': str(ARTIFACTS_DIR / 'sample_boxes.png'),
        'qualitative_predictions_png': str(ARTIFACTS_DIR / 'qualitative_predictions.png')
    }
}
manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))